In [4]:
"""
DART 한국 상장사 재무제표 수집 및 DB 업로드 시스템
- 안정적인 API 호출 및 에러 핸들링
- 배치 처리로 메모리 효율화
- FDR 기반 현재 상장 일반주식 필터링
- 최적화된 데이터 수집 속도
"""

import sys
import os
import io
import time
import zipfile
import datetime as dt
import random
import gc
import traceback
import logging
from pathlib import Path
from typing import List, Dict, Optional

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import pymysql
import xml.etree.ElementTree as ET
from tqdm import tqdm
import FinanceDataReader as fdr

# ============================================================
# 로깅 설정
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# ============================================================
# 경로 설정
# ============================================================

def add_repo_path():
    """프로젝트 루트를 자동 탐색하여 sys.path에 추가"""
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        current = Path.cwd()

    for parent in [current] + list(current.parents):
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            logger.info(f"Project root added: {parent}")
            return str(parent)

    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        logger.warning(f"Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("DATA 폴더를 찾을 수 없습니다.")


try:
    project_root = add_repo_path()
    from DATA.stock_invest_function import get_db_host
except ImportError:
    logger.warning("stock_invest_function import 실패 - DB 정보를 직접 설정해야 합니다")


# ============================================================
# 설정
# ============================================================

API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"
TARGET_TABLE = "korea_fs_data_from_DART"

# API 호출 설정
API_CALL_DELAY_MIN = 0.6  # 최소 대기 시간 (초)
API_CALL_DELAY_MAX = 1.0  # 최대 대기 시간 (초)
API_TIMEOUT = 30          # API 타임아웃 (초)
MAX_RETRIES = 3           # API 재시도 횟수

# 배치 처리 설정
BATCH_SIZE = 20           # 배치당 기업 수
BATCH_REST_TIME = 45      # 배치 간 휴식 시간 (초)


# ============================================================
# HTTP 세션 관리
# ============================================================

def create_robust_session():
    """재시도 로직과 User-Agent가 설정된 안정적인 HTTP 세션 생성"""
    session = requests.Session()

    retry_strategy = Retry(
        total=MAX_RETRIES,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=10,
        pool_maxsize=20,
    )

    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update({
        "User-Agent": "Mozilla/5.0 (DART-FS-Collector/2.0; +investment.research@example.com)",
        "Accept": "application/json"
    })

    return session


# ============================================================
# 1. 상장사 목록 로드
# ============================================================

def load_corp_code(api_key: str, cache_path="dart_corp_codes.csv") -> pd.DataFrame:
    """DART 기업 코드 목록 로드 (캐시 활용)"""
    if os.path.exists(cache_path):
        logger.info(f"캐시에서 기업 코드 로드: {cache_path}")
        return pd.read_csv(cache_path, dtype=str)

    logger.info("DART API에서 기업 코드 다운로드 중...")
    url = "https://opendart.fss.or.kr/api/corpCode.xml"

    try:
        resp = requests.get(url, params={"crtfc_key": api_key}, timeout=60)
        resp.raise_for_status()
    except requests.RequestException as e:
        logger.error(f"기업 코드 다운로드 실패: {e}")
        raise

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        xml_name = [x for x in z.namelist() if x.endswith(".xml")][0]
        with z.open(xml_name) as f:
            tree = ET.parse(f)

    rows = []
    for elem in tree.getroot().findall("list"):
        rows.append({
            "corp_code": elem.findtext("corp_code"),
            "corp_name": elem.findtext("corp_name"),
            "stock_code": elem.findtext("stock_code"),
        })

    df = pd.DataFrame(rows, dtype=str)
    df = df[df["stock_code"].notna() & (df["stock_code"] != "")]
    df.to_csv(cache_path, index=False, encoding="utf-8-sig")

    logger.info(f"기업 코드 저장 완료: {len(df)}개")
    return df


def get_corp_info(corp_df: pd.DataFrame, ticker: str) -> Optional[Dict[str, str]]:
    """종목코드로 기업 정보 조회"""
    ticker = str(ticker).zfill(6)
    row = corp_df.loc[corp_df["stock_code"] == ticker]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "corp_code": r["corp_code"],
        "corp_name": r["corp_name"],
        "stock_code": ticker,
    }


# ============================================================
# 2. 분기 재무제표 수집
# ============================================================

def get_dart_fs_quarterly(
    api_key: str,
    corp_code: str,
    start_year: int,
    end_year: int,
    fs_div: str = "CFS",
    verbose: bool = False,
    session: Optional[requests.Session] = None
) -> pd.DataFrame:
    """
    DART API로부터 분기별 재무제표 수집

    Args:
        api_key: DART API 키
        corp_code: 기업 고유번호
        start_year: 시작 연도
        end_year: 종료 연도
        fs_div: 재무제표 구분 (CFS: 연결, OFS: 개별)
        verbose: 상세 로그 출력 여부
        session: requests.Session 객체

    Returns:
        재무제표 DataFrame
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

    reprt_codes = {
        "11013": "Q1",
        "11012": "Q2",
        "11014": "Q3",
        "11011": "Q4",
    }

    all_rows = []
    api_error_count = 0

    if session is None:
        session = requests

    for year in range(start_year, end_year + 1):
        for rc, quarter in reprt_codes.items():
            params = {
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": str(year),
                "reprt_code": rc,
                "fs_div": fs_div,
                "page_no": 1,
                "page_count": 1000,
            }

            data = None
            for attempt in range(MAX_RETRIES):
                try:
                    r = session.get(url, params=params, timeout=API_TIMEOUT)
                    r.raise_for_status()
                    data = r.json()
                    break

                except requests.exceptions.Timeout:
                    if attempt < MAX_RETRIES - 1:
                        wait = 3 * (attempt + 1)
                        if verbose:
                            logger.warning(f"타임아웃 - {wait}초 후 재시도 [{attempt+1}/{MAX_RETRIES}]")
                        time.sleep(wait)
                    else:
                        if verbose:
                            logger.error(f"{year}-{quarter}: 최대 재시도 초과")
                        break

                except (requests.exceptions.RequestException, ValueError) as e:
                    if verbose:
                        logger.error(f"{year}-{quarter}: {str(e)[:100]}")
                    break

            if data is None:
                continue

            status = data.get("status")

            if verbose and status not in ["000", "013"]:
                logger.warning(f"{year}-{quarter}: status={status}, msg={data.get('message', 'N/A')}")

            if status not in ["000", "013"]:
                api_error_count += 1

            if status != "000":
                continue

            rows = data.get("list", [])
            for row in rows:
                row["reprt_code"] = rc
                row["quarter"] = quarter
                all_rows.append(row)

            # 랜덤 대기로 API 부하 분산
            time.sleep(random.uniform(API_CALL_DELAY_MIN, API_CALL_DELAY_MAX))

    if verbose and api_error_count > 0:
        logger.warning(f"API 오류 횟수: {api_error_count}")

    if not all_rows:
        return pd.DataFrame()

    raw = pd.DataFrame(all_rows)

    # 날짜 생성
    def make_date(row):
        y = int(row["bsns_year"])
        rc = row["reprt_code"]
        date_map = {
            "11013": (y, 3, 31),
            "11012": (y, 6, 30),
            "11014": (y, 9, 30),
            "11011": (y, 12, 31),
        }
        if rc in date_map:
            return pd.Timestamp(*date_map[rc])
        return pd.NaT

    raw["date"] = raw.apply(make_date, axis=1)
    return raw


# ============================================================
# 3. 계정 alias 정의
# ============================================================

ACCOUNT_ALIASES = {
    # 손익계산서
    "sales": ["매출", "수익(매출)", "매출액", "Revenue", "영업수익"],
    "cogs": ["매출원가", "Cost of sales"],
    "gross_profit": ["매출총이익", "Gross profit"],
    "op_income": ["영업이익", "Operating profit", "Operating income"],
    "net_income": ["당기순이익", "순이익", "Net income", "Profit"],

    # EPS
    "diluted_eps": ["희석주당순이익", "Diluted earnings", "희석 주당순이익"],

    # 자산
    "total_assets": ["자산총계", "총자산", "Total assets"],
    "current_assets": ["유동자산", "Current assets"],
    "cash": ["현금및현금성자산", "Cash and cash equivalents"],
    "st_financial": ["단기금융상품", "Short-term financial instruments"],
    "receivables": ["매출채권", "Trade receivables", "매출채권 및 기타유동채권"],
    "inventories": ["재고자산", "Inventories"],
    "noncurrent_assets": ["비유동자산", "Non-current assets"],
    "ppe": ["유형자산", "Property, plant and equipment"],
    "intangibles": ["무형자산", "Intangible assets"],

    # 부채/자본
    "total_liab": ["부채총계", "총부채", "Total liabilities"],
    "current_liab": ["유동부채", "Current liabilities"],
    "trade_payables": ["매입채무", "Trade payables", "매입채무 및 기타유동채무"],
    "st_borrowings": ["단기차입금", "Short-term borrowings"],
    "noncurrent_liab": ["비유동부채", "Non-current liabilities"],
    "equity": ["자본총계", "지배기업 소유주지분", "Total equity", "자본금"],

    # 현금흐름
    "op_cf": ["영업활동현금흐름", "영업활동으로 인한 현금흐름", "Operating cash flow"],
    "cce_increase": ["현금및현금성자산의 증가", "현금및현금성자산의증가(감소)", "Increase in cash"],
}


# ============================================================
# 4. 계정 추출 함수
# ============================================================

def _get_amount(fs_df: pd.DataFrame, key: str) -> Optional[float]:
    """계정 alias를 사용하여 금액 추출"""
    aliases = ACCOUNT_ALIASES.get(key, [])
    if not aliases:
        return None

    mask = pd.Series([False] * len(fs_df))

    for alias in aliases:
        m = fs_df["account_nm"].astype(str).str.contains(alias, na=False, regex=False)
        if "account_id" in fs_df.columns:
            m = m | fs_df["account_id"].astype(str).str.contains(alias, na=False, regex=False)
        mask = mask | m

    sub = fs_df[mask]
    if sub.empty:
        return None

    vals = pd.to_numeric(
        sub["thstrm_amount"].astype(str).str.replace(",", ""),
        errors="coerce"
    ).dropna()

    if vals.empty:
        return None

    return float(vals.sum())


def _get_dividend_paid(fs_df: pd.DataFrame) -> Optional[float]:
    """배당금 지급 금액 추출"""
    if "sj_div" in fs_df.columns:
        mask_cf = fs_df["sj_div"].astype(str).str.contains("CF", na=False)
    else:
        mask_cf = pd.Series([True] * len(fs_df))

    mask_div = (
        fs_df["account_id"].astype(str).str.contains("DividendsPaid", na=False)
        | fs_df["account_nm"].astype(str).str.contains("배당금 지급", na=False)
    )

    sub = fs_df[mask_cf & mask_div]

    if sub.empty:
        sub = fs_df[fs_df["account_nm"].astype(str).str.contains("배당", na=False)]

    if sub.empty:
        return None

    vals = pd.to_numeric(
        sub["thstrm_amount"].astype(str).str.replace(",", ""),
        errors="coerce"
    ).dropna()

    if vals.empty:
        return None

    return float(abs(vals.sum()))


# ============================================================
# 5. 분기별 재무지표 계산
# ============================================================

def compute_quarterly_indicators(fs_raw: pd.DataFrame) -> pd.DataFrame:
    """재무제표 원본 데이터에서 주요 지표 계산"""
    if fs_raw.empty:
        return pd.DataFrame()

    fs_raw["thstrm_amount"] = pd.to_numeric(
        fs_raw["thstrm_amount"].astype(str).str.replace(",", ""),
        errors="coerce"
    )

    records = []
    group_cols = ["corp_code", "corp_name", "date"]

    for (corp_code, corp_name, date), grp in fs_raw.groupby(group_cols):
        # 주요 계정 추출
        sales = _get_amount(grp, "sales")
        cogs = _get_amount(grp, "cogs")
        gross_profit = _get_amount(grp, "gross_profit")
        op_income = _get_amount(grp, "op_income")
        net_income = _get_amount(grp, "net_income")
        diluted_eps = _get_amount(grp, "diluted_eps")

        total_assets = _get_amount(grp, "total_assets")
        cur_assets = _get_amount(grp, "current_assets")
        cash = _get_amount(grp, "cash")
        st_fin = _get_amount(grp, "st_financial")
        recv = _get_amount(grp, "receivables")
        inv = _get_amount(grp, "inventories")
        nca = _get_amount(grp, "noncurrent_assets")
        ppe = _get_amount(grp, "ppe")
        intan = _get_amount(grp, "intangibles")

        total_liab = _get_amount(grp, "total_liab")
        cur_liab = _get_amount(grp, "current_liab")
        payables = _get_amount(grp, "trade_payables")
        st_borr = _get_amount(grp, "st_borrowings")
        noncur_liab = _get_amount(grp, "noncurrent_liab")
        equity = _get_amount(grp, "equity")

        op_cf = _get_amount(grp, "op_cf")
        cce_inc = _get_amount(grp, "cce_increase")
        dividend_paid = _get_dividend_paid(grp)

        # 안전한 나눗셈
        def safe_divide(a, b):
            if a is None or b in (None, 0):
                return None
            return float(a) / float(b)

        # 지표 계산
        indicators = {
            # 원본 계정
            "Sales": sales,
            "COGS": cogs,
            "Gross_Profit": gross_profit,
            "Operating_Income": op_income,
            "Net_Income": net_income,
            "Diluted_EPS": diluted_eps,

            "Total_Assets": total_assets,
            "Current_Assets": cur_assets,
            "Cash": cash,
            "ST_Financial": st_fin,
            "Receivables": recv,
            "Inventories": inv,
            "Noncurrent_Assets": nca,
            "PPE": ppe,
            "Intangibles": intan,

            "Total_Liabilities": total_liab,
            "Current_Liabilities": cur_liab,
            "Trade_Payables": payables,
            "ST_Borrowings": st_borr,
            "Noncurrent_Liabilities": noncur_liab,
            "Total_Equity": equity,

            "Operating_CF": op_cf,
            "CashEquivalents_Change": cce_inc,
            "Dividend_Paid": dividend_paid,

            # 재무비율
            "GPM": safe_divide(gross_profit, sales),
            "OPM": safe_divide(op_income, sales),
            "NIM": safe_divide(net_income, sales),
            "ROA": safe_divide(net_income, total_assets),
            "ROE": safe_divide(net_income, equity),
            "Debt_Ratio": safe_divide(total_liab, equity),
            "Current_Ratio": safe_divide(cur_assets, cur_liab),
            "OCF_to_Sales": safe_divide(op_cf, sales),
            "OCF_to_Assets": safe_divide(op_cf, total_assets),
            "Payout_Ratio": safe_divide(dividend_paid, net_income),
        }

        for indicator_name, value in indicators.items():
            if value is None:
                continue
            records.append({
                "date": date.date() if hasattr(date, 'date') else date,
                "company_name": corp_name,
                "ticker": corp_code,
                "indicator": indicator_name,
                "value": float(value),
            })

    return pd.DataFrame(records)


# ============================================================
# 6. DB 업로드
# ============================================================

def upload_indicators_to_db(
    df: pd.DataFrame,
    db_info: Dict,
    table_name: str = TARGET_TABLE
):
    """계산된 재무지표를 DB에 배치 업로드"""
    if df.empty:
        logger.warning("업로드할 데이터가 없습니다")
        return

    conn = None
    cursor = None

    try:
        conn = pymysql.connect(
            host=db_info["host"],
            port=db_info["port"],
            user=db_info["user"],
            password=db_info["password"],
            database=db_info["database"],
            charset="utf8mb4",
            autocommit=False,
        )

        sql = f"""
            INSERT INTO {table_name} (date, company_name, ticker, indicator, value)
            VALUES (%s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                company_name = VALUES(company_name),
                value = VALUES(value)
        """

        cursor = conn.cursor()
        rows = [tuple(row) for row in df.values]
        cursor.executemany(sql, rows)
        conn.commit()

        logger.info(f"DB 업로드 완료: {len(df)} rows")

    except pymysql.Error as e:
        if conn:
            conn.rollback()
        logger.error(f"DB 업로드 실패: {e}")
        raise

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()


# ============================================================
# 7. 전체 상장사 수집 메인 함수
# ============================================================

def collect_and_upload_all_companies(
    api_key: str,
    db_info: Dict,
    start_year: int = 2013,
    end_year: int = 2025,
    fs_div: str = "CFS",
    batch_size: int = BATCH_SIZE,
    table_name: str = TARGET_TABLE,
    verbose_first_n: int = 3,
    try_ofs_fallback: bool = True,
    batch_rest_time: int = BATCH_REST_TIME,
    use_fdr_filter: bool = True
):
    """
    모든 상장사의 재무데이터를 수집하고 배치 단위로 DB에 업로드

    Args:
        api_key: DART API 키
        db_info: DB 연결 정보
        start_year: 시작 연도
        end_year: 종료 연도
        fs_div: 재무제표 구분 (CFS/OFS)
        batch_size: 배치당 기업 수
        table_name: DB 테이블명
        verbose_first_n: 처음 N개 기업 상세 로그
        try_ofs_fallback: CFS 실패시 OFS 시도 여부
        batch_rest_time: 배치 간 휴식 시간
        use_fdr_filter: FDR 기반 현재 상장사 필터링 여부
    """
    session = create_robust_session()
    logger.info("HTTP 세션 생성 완료 (재시도 로직 + User-Agent)")

    # 1) DART 기업 목록 로드
    print("=" * 70)
    logger.info("[STEP 1] DART 기업 목록 로드 중...")

    corp_df = load_corp_code(api_key)

    # DART 상장사만 필터링
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"] != "") &
        (corp_df["stock_code"].str.strip() != "")
    ].copy()

    logger.info(f"DART 상장사 필터링 완료: {len(corp_df)}개")

    # 2) FDR 기반 현재 상장사 필터링
    if use_fdr_filter:
        logger.info("[STEP 1-2] FinanceDataReader로 현재 상장 종목 필터링...")
        try:
            fdr_df = fdr.StockListing("KRX")
            fdr_df["Code"] = fdr_df["Code"].astype(str).str.zfill(6)

            # ETF/ETN/REIT/SPAC 제거
            exclude_types = ["ETF", "ETN", "REIT", "SPAC"]

            if "Type" in fdr_df.columns:
                before = len(fdr_df)
                fdr_df = fdr_df[~fdr_df["Type"].isin(exclude_types)].copy()
                logger.info(f"Type 기반 필터링: {before}개 -> {len(fdr_df)}개")
            else:
                logger.warning("FDR 데이터에 'Type' 컬럼 없음 - Name 기반 필터 사용")
                pattern = r"ETF|ETN|리츠|리트|스팩|SPAC"
                before = len(fdr_df)
                fdr_df = fdr_df[~fdr_df["Name"].str.contains(pattern, case=False, na=False)].copy()
                logger.info(f"Name 기반 필터링: {before}개 -> {len(fdr_df)}개")

            fdr_codes = set(fdr_df["Code"].tolist())
            logger.info(f"FDR 현재 상장 일반 주식: {len(fdr_codes)}개")

            corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
            before = len(corp_df)
            corp_df = corp_df[corp_df["stock_code"].isin(fdr_codes)].copy()
            logger.info(f"FDR 필터링 완료: {before}개 -> {len(corp_df)}개")

        except Exception as e:
            logger.error(f"FDR 필터링 실패: {e}")
            logger.warning("FDR 필터를 건너뛰고 DART 목록만 사용")

    total_companies = len(corp_df)
    logger.info(f"최종 대상 기업: {total_companies}개")

    print("\n[상장사 샘플]")
    print(corp_df[["corp_name", "stock_code"]].head(10))
    print()

    # 3) 통계 변수 초기화
    success_count = 0
    fail_count = 0
    no_data_count = 0
    cfs_to_ofs_count = 0
    failed_companies = []
    processed_count = 0

    # 4) 배치 처리
    print("=" * 70)
    logger.info("[STEP 2] 재무데이터 수집 시작")
    logger.info(f"설정: 배치={batch_size}, 휴식={batch_rest_time}초, "
                f"API대기={API_CALL_DELAY_MIN}~{API_CALL_DELAY_MAX}초")
    print("=" * 70)

    total_batches = (total_companies + batch_size - 1) // batch_size

    for batch_idx in range(0, total_companies, batch_size):
        batch_end = min(batch_idx + batch_size, total_companies)
        batch_df = corp_df.iloc[batch_idx:batch_end]
        current_batch = batch_idx // batch_size + 1

        print(f"\n{'='*70}")
        print(f"[BATCH {current_batch}/{total_batches}] "
              f"처리 중: {batch_idx+1}~{batch_end}/{total_companies}")
        print(f"{'='*70}")

        batch_indicators = []

        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc="기업 처리"):
            ticker = row["stock_code"]
            corp_name = row["corp_name"]
            corp_code = row["corp_code"]
            processed_count += 1

            verbose = (processed_count <= verbose_first_n)

            if verbose:
                print(f"\n  [상세로그 {processed_count}] {corp_name} ({ticker})")

            try:
                # CFS 우선 시도
                fs_raw = get_dart_fs_quarterly(
                    api_key=api_key,
                    corp_code=corp_code,
                    start_year=start_year,
                    end_year=end_year,
                    fs_div=fs_div,
                    verbose=verbose,
                    session=session
                )

                # CFS 실패시 OFS 대체
                if fs_raw.empty and try_ofs_fallback and fs_div == "CFS":
                    if verbose:
                        print(f"    -> CFS 없음, OFS 시도...")

                    fs_raw = get_dart_fs_quarterly(
                        api_key=api_key,
                        corp_code=corp_code,
                        start_year=start_year,
                        end_year=end_year,
                        fs_div="OFS",
                        verbose=verbose,
                        session=session
                    )

                    if not fs_raw.empty:
                        cfs_to_ofs_count += 1
                        if verbose:
                            print(f"    -> OFS 데이터 확보!")

                if fs_raw.empty:
                    no_data_count += 1
                    if verbose:
                        print(f"    -> 데이터 없음")
                    continue

                # 메타 정보 추가
                fs_raw["corp_code"] = corp_code
                fs_raw["corp_name"] = corp_name
                fs_raw["ticker"] = ticker

                # 지표 계산
                df_ind = compute_quarterly_indicators(fs_raw)

                if not df_ind.empty:
                    df_ind["ticker"] = ticker
                    batch_indicators.append(df_ind)
                    success_count += 1

                    if verbose or success_count <= 3:
                        print(f"  ✓ [{ticker}] {corp_name}: {len(df_ind)}개 지표")
                else:
                    no_data_count += 1
                    if verbose:
                        print(f"    -> 지표 계산 결과 없음")

                del fs_raw, df_ind

            except Exception as e:
                fail_count += 1
                error_msg = str(e)[:200]
                failed_companies.append({
                    "ticker": ticker,
                    "corp_name": corp_name,
                    "error": error_msg
                })

                print(f"  X [{ticker}] {corp_name}: 오류")
                if verbose:
                    print(f"     -> {error_msg}")

                # 연속 오류시 대기
                if fail_count % 5 == 0:
                    logger.warning(f"연속 오류 {fail_count}회 - 20초 대기")
                    time.sleep(20)

        # 배치 업로드
        if batch_indicators:
            print(f"\n{'='*70}")
            logger.info(f"[BATCH UPLOAD] 배치 {current_batch} DB 업로드 중...")

            try:
                batch_combined = pd.concat(batch_indicators, ignore_index=True)
                upload_indicators_to_db(batch_combined, db_info, table_name)
                logger.info(f"배치 업로드 완료: {len(batch_combined)} rows")
                del batch_combined

            except Exception as e:
                logger.error(f"배치 업로드 실패: {e}")
                traceback.print_exc()
        else:
            logger.warning(f"배치 {current_batch}: 업로드할 데이터 없음")

        del batch_indicators
        gc.collect()

        print(f"{'='*70}")
        print(f"[진행] 성공: {success_count} | 데이터없음: {no_data_count} | 실패: {fail_count}")
        if cfs_to_ofs_count > 0:
            print(f"[OFS 대체] {cfs_to_ofs_count}개")
        print(f"{'='*70}")

        # 배치 간 휴식
        if batch_end < total_companies:
            logger.info(f"다음 배치 전 {batch_rest_time}초 휴식...")
            time.sleep(batch_rest_time)

    session.close()

    # 5) 최종 결과
    print("\n" + "=" * 70)
    print("[최종 결과]")
    print("=" * 70)
    print(f"✓ 성공: {success_count}개")
    print(f"- 데이터없음: {no_data_count}개")
    print(f"X 실패: {fail_count}개")
    if cfs_to_ofs_count > 0:
        print(f"-> CFS->OFS 대체: {cfs_to_ofs_count}개")
    print(f"총 처리: {total_companies}개")
    if total_companies > 0:
        print(f"성공률: {success_count/total_companies*100:.1f}%")
        valid_attempts = success_count + no_data_count
        if valid_attempts > 0:
            print(f"데이터 확보율: {success_count/valid_attempts*100:.1f}%")

    # 6) 실패 목록
    if failed_companies:
        print("\n" + "=" * 70)
        print("[실패한 기업]")
        print("=" * 70)
        for item in failed_companies[:20]:
            print(f"  [{item['ticker']}] {item['corp_name']}")
            print(f"    {item['error'][:100]}")

        if len(failed_companies) > 20:
            print(f"  ... 외 {len(failed_companies)-20}개")

        failed_df = pd.DataFrame(failed_companies)
        failed_csv = f"failed_companies_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")
        logger.info(f"실패 목록 저장: {failed_csv}")

    print("=" * 70)
    print("완료!")
    print("=" * 70)


# ============================================================
# 8. 실행 예제
# ============================================================

# if __name__ == "__main__":
#     # DB 정보 설정
#     try:
#         db_info = get_db_host()
#     except:
#         db_info = {
#             "host": "192.168.0.230",
#             "port": 3307,
#             "user": "investar",
#             "password": "PASSWORD",  # 실제 비밀번호로 변경 필요
#             "database": "investar",
#         }
#         logger.warning("get_db_host() 실패 - 기본 DB 설정 사용")
#
#     # 데이터 수집 및 업로드 실행
#     collect_and_upload_all_companies(
#         api_key=API_KEY,
#         db_info=db_info,
#         start_year=2013,
#         end_year=2025,
#         fs_div="CFS",
#         batch_size=BATCH_SIZE,
#         table_name=TARGET_TABLE,
#         verbose_first_n=3,
#         try_ofs_fallback=True,
#         batch_rest_time=BATCH_REST_TIME,
#         use_fdr_filter=True
#     )

2025-11-18 22:43:51 [INFO] Project root added: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [5]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

collect_and_upload_all_companies(
        api_key=API_KEY,
        db_info=db_info,
        start_year=2013,
        end_year=2025,
        fs_div="CFS",
        batch_size=BATCH_SIZE,
        table_name=TARGET_TABLE,
        verbose_first_n=3,
        try_ofs_fallback=True,
        batch_rest_time=BATCH_REST_TIME,
        use_fdr_filter=True
    )

2025-11-18 22:43:53 [INFO] HTTP 세션 생성 완료 (재시도 로직 + User-Agent)
2025-11-18 22:43:53 [INFO] [STEP 1] DART 기업 목록 로드 중...
2025-11-18 22:43:53 [INFO] 캐시에서 기업 코드 로드: dart_corp_codes.csv
2025-11-18 22:43:53 [INFO] DART 상장사 필터링 완료: 3906개
2025-11-18 22:43:53 [INFO] [STEP 1-2] FinanceDataReader로 현재 상장 종목 필터링...


2025-11-18 22:43:54 [WARNING] FDR 데이터에 'Type' 컬럼 없음 - Name 기반 필터 사용
2025-11-18 22:43:54 [INFO] Name 기반 필터링: 2881개 -> 2780개
2025-11-18 22:43:54 [INFO] FDR 현재 상장 일반 주식: 2780개
2025-11-18 22:43:54 [INFO] FDR 필터링 완료: 3906개 -> 2662개
2025-11-18 22:43:54 [INFO] 최종 대상 기업: 2662개
2025-11-18 22:43:54 [INFO] [STEP 2] 재무데이터 수집 시작
2025-11-18 22:43:54 [INFO] 설정: 배치=20, 휴식=45초, API대기=0.6~1.0초



[상장사 샘플]
      corp_name stock_code
36636       GRT     900290
41823       로스웰     900260
48350    맥쿼리인프라     088980
48630   크리스탈신소재     900250
51444        우진     105840
51563      인화정공     101930
51576      대원산업     005710
51577        대동     000490
51805   삼화콘덴서공업     001820
51855       유니온     000910


[BATCH 1/134] 처리 중: 1~20/2662


기업 처리:   0%|          | 0/20 [00:00<?, ?it/s]2025-11-18 22:43:54 [WARNING] 2013-Q1: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:43:54 [WARNING] 2013-Q2: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:43:54 [WARNING] 2013-Q3: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:43:54 [WARNING] 2013-Q4: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:43:54 [WARNING] 2014-Q1: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:43:54 [WARNING] 2014-Q2: status=020, msg=사용한도를 초과하였습니다.



  [상세로그 1] GRT (900290)


...

    -> CFS 없음, OFS 시도...


2025-11-18 22:44:00 [WARNING] 2015-Q2: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2015-Q3: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2015-Q4: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2016-Q1: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2016-Q2: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2016-Q3: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2016-Q4: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2017-Q1: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2017-Q2: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:00 [WARNING] 2017-Q3: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:01 [WARNING] 2017-Q4: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:01 [WARNING] 2018-Q1: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:01 [WARNING] 2018-Q2: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:01 [WARNING] 2018-Q3: status=020, msg=사용한도를 초과하였습니다.
2025-11-18 22:44:01 

    -> 데이터 없음


기업 처리:  25%|██▌       | 5/20 [00:13<00:39,  2.66s/it]


KeyboardInterrupt: 